In [1]:
# ============================================================
# Jester_02_TFIDF_Baseline.ipynb
# Part 1 — Load Jester edges + jokes
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Folder where i saved Notebook 1 outputs
JESTER_DIR = Path.home() / "Downloads" / "Joke_Project_Diss2026" / "Jester"

edges_path = JESTER_DIR / "jester_edges_long.csv"
jokes_path = JESTER_DIR / "jester_jokes.csv"

edges = pd.read_csv(edges_path)
jokes_df = pd.read_csv(jokes_path)

print("edges shape:", edges.shape)
print("jokes_df shape:", jokes_df.shape)

edges.head()


edges shape: (1810455, 3)
jokes_df shape: (100, 2)


,user_id,joke_id,rating
0,0,1,-7.82
1,0,2,8.79
2,0,3,-9.66
3,0,4,-8.16
4,0,5,-7.52


In [2]:
# ============================================================
# Part 2 — Make implicit likes + filter sparse users
# ============================================================

# Like = rating > 0 (common Jester setup)
edges["label"] = (edges["rating"] > 0).astype(int)

# Only keep positive interactions
pos = edges[edges["label"] == 1][["user_id", "joke_id"]].copy()

print("Positive interactions:", len(pos))
print("Unique users with >=1 like:", pos["user_id"].nunique())

# Filter users with very few likes (stabilises evaluation)
MIN_LIKES_PER_USER = 10

user_like_counts = pos["user_id"].value_counts()
keep_users = user_like_counts[user_like_counts >= MIN_LIKES_PER_USER].index

pos = pos[pos["user_id"].isin(keep_users)].reset_index(drop=True)

print("\nAfter filtering:")
print("Positive interactions:", len(pos))
print("Users:", pos["user_id"].nunique(), "| Items:", pos["joke_id"].nunique())


Positive interactions: 1077115
Unique users with >=1 like: 24913

After filtering:
Positive interactions: 1073324
Users: 24271 | Items: 100


In [3]:
# ============================================================
# Part 3 — Train/Test split per user
# ============================================================

rng = np.random.default_rng(42)

TEST_PER_USER = 2  # hold out 2 liked jokes per user

train_rows, test_rows = [], []

for user_id, grp in pos.groupby("user_id"):
    jokes = grp["joke_id"].to_numpy()

    if len(jokes) <= TEST_PER_USER:
        continue

    test_jokes = rng.choice(jokes, size=TEST_PER_USER, replace=False)
    train_jokes = np.setdiff1d(jokes, test_jokes)

    train_rows.extend([(user_id, j) for j in train_jokes])
    test_rows.extend([(user_id, j) for j in test_jokes])

train_pos = pd.DataFrame(train_rows, columns=["user_id", "joke_id"])
test_pos  = pd.DataFrame(test_rows,  columns=["user_id", "joke_id"])

print("Train positives:", len(train_pos))
print("Test positives:", len(test_pos))
print("Users in test:", test_pos["user_id"].nunique())


Train positives: 1024782
Test positives: 48542
Users in test: 24271


In [4]:
# ============================================================
# Part 4 — Build TF-IDF vectors for jokes
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Ensure joke_id order matches TF-IDF row order
jokes_df = jokes_df.sort_values("joke_id").reset_index(drop=True)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X = vectorizer.fit_transform(jokes_df["joke_text"].astype(str))

print("TF-IDF matrix shape:", X.shape)


TF-IDF matrix shape: (100, 1395)


In [8]:
# ============================================================
# Part 5 — Recommend Top-K with TF-IDF user profile
# ============================================================

# user -> set of train liked jokes
train_likes = train_pos.groupby("user_id")["joke_id"].apply(set).to_dict()

# joke_id <-> row index in TF-IDF matrix
all_joke_ids = jokes_df["joke_id"].to_numpy()
joke_id_to_idx = {jid: i for i, jid in enumerate(all_joke_ids)}

def recommend_tfidf(user_id: int, k: int = 10):
    """Return top-k joke recommendations based on TF-IDF similarity."""
    liked = train_likes.get(user_id, set())
    if len(liked) == 0:
        return []

    liked_idx = [joke_id_to_idx[j] for j in liked if j in joke_id_to_idx]

    # Average profile (convert np.matrix -> ndarray)
    user_vec = np.asarray(X[liked_idx].mean(axis=0))

    sims = cosine_similarity(user_vec, X).ravel()

    # Don’t recommend already-liked jokes
    for j in liked:
        if j in joke_id_to_idx:
            sims[joke_id_to_idx[j]] = -1.0

    top_idx = np.argsort(sims)[::-1][:k]
    return [(int(all_joke_ids[i]), float(sims[i])) for i in top_idx]

# Quick demo
demo_user = int(test_pos["user_id"].iloc[0])
print("Demo user:", demo_user)
print(recommend_tfidf(demo_user, k=5))




Demo user: 0
[(93, 0.18398252417471558), (44, 0.15283767971117168), (76, 0.13810951241921732), (46, 0.12445794163250257), (86, 0.12165813743958694)]


In [9]:
# ============================================================
# Part 6 — Evaluate TF-IDF baseline (Precision/Recall/NDCG)
# ============================================================

def ndcg_at_k(recommended_ids, true_ids, k):
    """NDCG@k for binary relevance."""
    recommended_ids = recommended_ids[:k]

    dcg = 0.0
    for i, jid in enumerate(recommended_ids):
        rel = 1.0 if jid in true_ids else 0.0
        dcg += rel / np.log2(i + 2)

    ideal_hits = min(len(true_ids), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))

    return dcg / idcg if idcg > 0 else 0.0

def evaluate_tfidf(k=10, max_users=2000):
    """Evaluate on up to max_users users (speed)."""
    users = test_pos["user_id"].unique()
    users = users[:min(len(users), max_users)]

    test_truth = test_pos.groupby("user_id")["joke_id"].apply(set).to_dict()

    precisions, recalls, ndcgs = [], [], []

    for u in users:
        truth = test_truth.get(int(u), set())

        recs = recommend_tfidf(int(u), k=k)
        rec_ids = [jid for jid, _ in recs]

        hits = sum(1 for jid in rec_ids if jid in truth)
        precisions.append(hits / k)
        recalls.append(hits / len(truth) if len(truth) else 0.0)
        ndcgs.append(ndcg_at_k(rec_ids, truth, k))

    return {
        "K": k,
        "UsersEvaluated": int(len(users)),
        "Precision@K": float(np.mean(precisions)),
        "Recall@K": float(np.mean(recalls)),
        "NDCG@K": float(np.mean(ndcgs)),
    }

results_5  = evaluate_tfidf(k=5)
results_10 = evaluate_tfidf(k=10)

print(results_5)
print(results_10)


{'K': 5, 'UsersEvaluated': 2000, 'Precision@K': 0.0605, 'Recall@K': 0.15125, 'NDCG@K': 0.10619627997583679}
{'K': 10, 'UsersEvaluated': 2000, 'Precision@K': 0.054299999999999994, 'Recall@K': 0.2715, 'NDCG@K': 0.15336266038360685}
